|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 8:</h2>|<h1>The Capstone<h1>|
|<h2>Section:</h2>|<h1>One engine<h1>|
|<h2>Lecture:</h2>|<h1><b>The flat batch: a host for your kernels<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import sys
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

# The parts never met

Stages 06 to 20 built the parts of an engine. Each part passed its checks
alone, on random tensors or in a simulation. Not one of them ran inside a
model.

The reason is the host. The HuggingFace model grows its KV cache by
concatenation, and it hides the attention call. A paged kernel has nowhere to
live. This notebook measures what that host costs, and then shows the host
that the capstone uses instead.

In [2]:
from tvllm import load_model

model = load_model('Qwen/Qwen3-1.7B')
config = model.config
print(f'{config.name}: {config.num_layers} layers, {config.num_heads} query heads, '
      f'{config.num_kv_heads} KV heads, head_dim {config.head_dim}')
print(f'weight bytes read in each decode step: {model.weight_bytes()/1e9:.2f} GB')
print(f'KV bytes for each token:               {model.kv_bytes_per_token()/1024:.0f} KiB')

/home/venugopalan/vllm-from-scratch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 17580.85it/s]

Qwen/Qwen3-1.7B: 28 layers, 16 query heads, 8 KV heads, head_dim 128
weight bytes read in each decode step: 3.44 GB
KV bytes for each token:               112 KiB


### What a padded batch costs

The HF model takes a rectangle: (batch, length). So you must pad a batch of
ragged prompts to the longest one, and the padding costs the same arithmetic
as real tokens. Count the waste for a batch that a real server can see.

In [3]:
lengths = [12, 40, 300, 25, 700, 90, 8, 160]
num_real_tokens = sum(lengths)
num_padded_tokens = len(lengths) * max(lengths)
print(f'real tokens {num_real_tokens}, padded rectangle {num_padded_tokens}: '
      f'{num_padded_tokens / num_real_tokens:.1f}x the work')

real tokens 1335, padded rectangle 5600: 4.2x the work


### Now time it

Run the same prompts two ways through the same model: as one flat batch, and
as the padded rectangle. Both use the same attention code, so the difference
is only the extra tokens.

In [4]:
from tvllm.model import attend_to_context

def segments_for(lengths):
    """-> the (start, length) of each sequence in the flat batch."""
    starts = np.cumsum([0] + list(lengths[:-1]))
    return [(int(start), length) for start, length in zip(starts, lengths)]

class DenseAttention:
    """Causal attention for the sequences of a flat batch, with no cache."""
    def __init__(self, segments):
        self.segments = segments

    def __call__(self, layer, query, key, value):
        num_tokens, num_heads, head_dim = query.shape
        output = torch.empty(num_tokens, num_heads * head_dim,
                             dtype=query.dtype, device=query.device)
        for start, length in self.segments:
            rows = slice(start, start + length)
            output[rows] = attend_to_context(query[rows], key[rows].transpose(0, 1),
                                             value[rows].transpose(0, 1))
        return output

def prefill_ms(lengths):
    segments = segments_for(lengths)
    token_ids = torch.randint(0, 1000, (sum(lengths),), device='cuda')
    positions = torch.cat([torch.arange(length, device='cuda') for length in lengths])
    last_rows = torch.tensor([start + length - 1 for start, length in segments],
                             device='cuda')
    attention = DenseAttention(segments)
    return cudalib.bench_ms(lambda: model.forward(token_ids, positions, attention, last_rows),
                            iters=10, warmup=3, best_of=3)

flat_ms = prefill_ms(lengths)
padded_ms = prefill_ms([max(lengths)] * len(lengths))
print(f'flat {flat_ms:.1f} ms, padded {padded_ms:.1f} ms: the padded batch takes '
      f'{padded_ms / flat_ms:.1f}x the time, for '
      f'{num_padded_tokens / num_real_tokens:.1f}x the tokens')

flat 151.5 ms, padded 721.0 ms: the padded batch takes 4.8x the time, for 4.2x the tokens


### Read the two ratios together

The time ratio follows the token ratio, because a prefill of this size waits
on arithmetic, and padding is arithmetic. The time ratio is a little larger,
because attention over a long padded row grows faster than linearly. The work
that the padding adds does not depend on the card. Only the milliseconds do.

The flat batch removes that waste completely. It also removes a second limit:
in a flat batch, one step can hold one-token decodes next to long prefill
chunks. A rectangle cannot hold both.

# The host that the capstone uses

`tvllm/model.py` is the same transformer with two changes:

- It takes a FLAT batch. All the tokens of all the sequences sit in one (T,)
  tensor. There is no padding and no batch dimension.
- It does not do attention itself. For each layer it calls a backend,
  `attention(layer, query, key, value)`, and the backend owns the cache.

Below is the smallest backend that serves a flat batch of several sequences.
It keeps one growing K and V for each sequence. Stage 21 replaces it with your
paged cache and your kernels.

In [5]:
class TinyFlatBackend:
    """Several sequences in one flat batch, with a growing K and V for each.
    segments[i] is the (start, length) of sequence i in this step."""
    def __init__(self, num_layers, num_seqs):
        self.keys = [[None] * num_seqs for _ in range(num_layers)]
        self.values = [[None] * num_seqs for _ in range(num_layers)]
        self.segments = []

    def _append(self, layer, seq, key, value):
        cached_keys, cached_values = self.keys[layer][seq], self.values[layer][seq]
        if cached_keys is None:
            self.keys[layer][seq], self.values[layer][seq] = key, value
        else:
            self.keys[layer][seq] = torch.cat([cached_keys, key])
            self.values[layer][seq] = torch.cat([cached_values, value])

    def __call__(self, layer, query, key, value):
        num_tokens, num_heads, head_dim = query.shape
        output = torch.empty(num_tokens, num_heads * head_dim,
                             dtype=query.dtype, device=query.device)
        for seq, (start, length) in enumerate(self.segments):
            rows = slice(start, start + length)
            self._append(layer, seq, key[rows], value[rows])
            output[rows] = attend_to_context(query[rows],
                                             self.keys[layer][seq].transpose(0, 1),
                                             self.values[layer][seq].transpose(0, 1))
        return output

prompts = ['The capital of France is', 'def fibonacci(n):', 'In 1969, humans first']
prompt_ids = [model.tokenizer(prompt).input_ids for prompt in prompts]

def run_together(num_steps=8):
    """Greedy decode of all prompts in one flat batch. -> the new tokens."""
    backend = TinyFlatBackend(config.num_layers, len(prompt_ids))
    generated = [[] for _ in prompt_ids]
    feeds = [list(ids) for ids in prompt_ids]      # the tokens to process in this step
    next_positions = [0] * len(prompt_ids)
    for _ in range(num_steps):
        flat_ids, positions, last_rows, backend.segments = [], [], [], []
        for seq, feed in enumerate(feeds):
            backend.segments.append((len(flat_ids), len(feed)))
            flat_ids += feed
            positions += range(next_positions[seq], next_positions[seq] + len(feed))
            last_rows.append(len(flat_ids) - 1)
        logits = model.forward(torch.tensor(flat_ids, device='cuda'),
                               torch.tensor(positions, device='cuda'), backend,
                               torch.tensor(last_rows, device='cuda'))
        for seq, row in enumerate(logits):
            next_positions[seq] += len(feeds[seq])
            token = int(row.argmax())
            generated[seq].append(token)
            feeds[seq] = [token]
    return generated

together = run_together()
for prompt, tokens in zip(prompts, together):
    print(f'{prompt!r:34} -> {model.tokenizer.decode(tokens)!r}')

'The capital of France is'         -> ' Paris. The capital of the United States'
'def fibonacci(n):'                -> '\n    if n <= 0:\n'
'In 1969, humans first'            -> ' set foot on the Moon. In '


### One flat batch, three prompts of three lengths

The first step fed three prompts of different lengths in ONE tensor, with no
padding. Every later step fed one token for each sequence. The model never
saw a batch dimension.

Now check that each answer is the same as the answer of the prompt alone.
In bf16 the reduction order changes with the batch, so a near tie can flip a
late token. Compare the first tokens.

In [6]:
alone = []
for ids in prompt_ids:
    backend = TinyFlatBackend(config.num_layers, 1)
    backend.segments = [(0, len(ids))]
    logits = model.forward(torch.tensor(ids, device='cuda'),
                           torch.arange(len(ids), device='cuda'), backend,
                           torch.tensor([len(ids) - 1], device='cuda'))
    alone.append(int(logits[0].argmax()))
print('first token, alone vs together:',
      [(alone_token, tokens[0]) for alone_token, tokens in zip(alone, together)])
print('all equal:', all(alone_token == tokens[0]
                        for alone_token, tokens in zip(alone, together)))

first token, alone vs together: [(12095, 12095), (198, 198), (738, 738)]
all equal: True


### What you take into stage 21

- The host must accept a flat batch, or a mixed step of decodes and prefill
  chunks cannot exist.
- The cache must belong to you, and not to the model. Then you can page it,
  and a new token costs one slot write, and not a copy of the cache.

`TinyFlatBackend` still concatenates, still loops over sequences in Python,
and still owns one tensor for each sequence. Stage 21 replaces all three with
your block allocator, your stage 08 write and your stage 08c kernel.

    ./vc guide 21